# Módulo 3: Feature Engineering y Preparación de Datos

## 1. División Train / Test y Diseño Muestral

El primer paso crítico antes de aplicar cualquier técnica de *Machine Learning* es establecer un diseño muestral robusto. Dividir el dataset en conjuntos de Entrenamiento (*Train*) y Prueba (*Test*) garantiza que nuestro modelo sea evaluado sobre datos invisibles, midiendo su verdadera capacidad de generalización.

### 1.1 Definición del Target (¿Qué vamos a predecir?)
Para este modelo predictivo, adoptaremos el enfoque de un **Modelo Look-Alike**. 
* **La Variable Objetivo ($y$):** Será la columna `Cluster` que descubrimos en el Módulo 2 mediante Análisis Multivariante (MCA) y K-Means.
* **El Objetivo de Negocio:** Entrenar un algoritmo capaz de predecir automáticamente si un nuevo visitante de la web pertenece al **Clúster 1 ("Motor Comercial", $y=1$)** o al **Clúster 0 ("Consumidor Cauto", $y=0$)**, basándose en su huella demográfica y tecnológica. Esto permitirá a la empresa activar pasarelas de pago y estrategias de UX personalizadas en tiempo real.

*(Nota Metodológica: Haber definido el Target mediante clustering sobre el dataset completo en la fase de diseño **no constituye Data Leakage**, ya que el clúster actúa como nuestra "Verdad Fundamental" o etiqueta real. El Leakage se evitará asegurando que las variables derivadas del MCA no participen como variables predictoras $X$ en el entrenamiento).*

### 1.2 Estrategia de Partición: La Llave Compuesta
Un muestreo aleatorio simple (*Random Split*) asume que la distribución de las clases se mantendrá por azar. En problemas de negocio con sub-segmentos críticos, esto es un riesgo inaceptable.

Para blindar nuestra partición, diseñamos una **Llave Compuesta de Estratificación**:
> `Llave_Estratificacion = Cluster + who_pays_for_access_work`

**¿Por qué esta llave?** No solo queremos garantizar que la proporción del Target (55.6% vs 44.4%) se mantenga intacta en Train y Test. También queremos asegurar que la proporción de usuarios "Corporativos" (nuestra variable estratégica aislada en el Módulo 2) se distribuya equitativamente dentro de cada clúster, evitando que por azar el conjunto de prueba se llene de oficinistas y distorsione las métricas de evaluación.

### 1.3 Análisis de la Comparativa de Robustez (Aleatorio vs. Estratificado)
Al ejecutar el experimento cruzado con una partición 80/20, la tabla de resultados demuestra la superioridad del método estratificado:

* **El Peligro del Azar:** En el método aleatorio, observamos una deriva (*drift*) en el conjunto de Test. El Clúster 1 cayó de 55.62% a **54.40%**, y la representación corporativa se infló al **27.35%**. Evaluar el modelo en este escenario penalizaría injustamente su rendimiento.
* **Precisión Quirúrgica:** Bajo nuestro método estratificado por llave compuesta, el conjunto de Test calcó la realidad casi a la perfección: **55.59%** para el Clúster 1 y **26.85%** para el acceso corporativo. 

**💡 Conclusión del Muestreo:**
Se ha garantizado un entorno de modelado matemáticamente idéntico al universo real. La llave auxiliar ha sido eliminada para evitar fugas de información, y los artefactos finales (`train_ecommerce.csv` y `test_ecommerce.csv`) han sido guardados exitosamente. Todo el análisis exploratorio y la ingeniería de variables a partir de este punto **se ejecutarán estrictamente sobre el conjunto de Entrenamiento (`Train`)** para preservar la pureza del conjunto de Prueba (`Test`).

In [2]:
# ==============================================================================
# CARGAR DATOS PROCESADOS DEL MÓDULO ANTERIOR
# ==============================================================================
import pandas as pd

# Agregamos el nombre del archivo al final de la ruta: /base_modulo2_procesada.csv
ruta_archivo = 'C:/Users/Matiu/Documents/GitHub/Data-Science/Futura/Módulo 2/Exams/Caso17_Usuarios_Internet/data_processed/base_modulo2_procesada.csv'

# 'Convocamos' la base analítica completa que dejó el Módulo 2
df_analitico = pd.read_csv(ruta_archivo, index_col=0)

print(f"📦 Dataset convocado exitosamente: {df_analitico.shape[0]} filas × {df_analitico.shape[1]} columnas.")
print("Distribución del Target (Cluster):")
display(df_analitico['Cluster'].value_counts(normalize=True) * 100)

📦 Dataset convocado exitosamente: 10108 filas × 62 columnas.
Distribución del Target (Cluster):


Cluster
1    55.619311
0    44.380689
Name: proportion, dtype: float64

In [4]:
# ==============================================================================
# 1.1 DISEÑO MUESTRAL: CREACIÓN DE LA LLAVE COMPUESTA Y SPLIT TRAIN/TEST
# ==============================================================================
from sklearn.model_selection import train_test_split
import numpy as np

print("🔑 1. CREANDO LLAVE COMPUESTA DE ESTRATIFICACIÓN...")
# Creamos una llave que cruce nuestro Target (Cluster) con nuestra variable de referencia
df_analitico['Llave_Estratificacion'] = df_analitico['Cluster'].astype(str) + "_Lab:" + df_analitico['who_pays_for_access_work'].astype(str)

print("Distribución de la Llave Compuesta en el universo total:")
display(df_analitico['Llave_Estratificacion'].value_counts(normalize=True).round(4) * 100)
print("-" * 75)

# ------------------------------------------------------------------------------
# EXPERIMENTO A: SPLIT ALEATORIO (El error común de los novatos)
# ------------------------------------------------------------------------------
train_random, test_random = train_test_split(df_analitico, test_size=0.20, random_state=42)

# ------------------------------------------------------------------------------
# EXPERIMENTO B: SPLIT ESTRATIFICADO (La estrategia Senior)
# ------------------------------------------------------------------------------
train_strat, test_strat = train_test_split(
    df_analitico, 
    test_size=0.20, 
    random_state=42, 
    stratify=df_analitico['Llave_Estratificacion']
)

print("\n⚖️ 2. COMPARATIVA DE ROBUSTEZ MUESTRAL (MÉTODO ALEATORIO VS ESTRATIFICADO)")
# Vamos a crear una tabla que compare cómo quedó la proporción del Clúster 1 (Target)
# y de los usuarios corporativos en ambos métodos de muestreo.

comparativa = pd.DataFrame({
    'Métrica': [
        '% Clúster 1 (Motor Comercial)', 
        '% Clúster 0 (Consumidor Cauto)',
        '% Acceso Corporativo (Sí)'
    ],
    'Universo Real': [
        (df_analitico['Cluster'] == 1).mean() * 100,
        (df_analitico['Cluster'] == 0).mean() * 100,
        (df_analitico['who_pays_for_access_work'] == 'Si').mean() * 100
    ],
    'Train Aleatorio': [
        (train_random['Cluster'] == 1).mean() * 100,
        (train_random['Cluster'] == 0).mean() * 100,
        (train_random['who_pays_for_access_work'] == 'Si').mean() * 100
    ],
    'Test Aleatorio': [
        (test_random['Cluster'] == 1).mean() * 100,
        (test_random['Cluster'] == 0).mean() * 100,
        (test_random['who_pays_for_access_work'] == 'Si').mean() * 100
    ],
    'Train Estratificado (Nuestro)': [
        (train_strat['Cluster'] == 1).mean() * 100,
        (train_strat['Cluster'] == 0).mean() * 100,
        (train_strat['who_pays_for_access_work'] == 'Si').mean() * 100
    ],
    'Test Estratificado (Nuestro)': [
        (test_strat['Cluster'] == 1).mean() * 100,
        (test_strat['Cluster'] == 0).mean() * 100,
        (test_strat['who_pays_for_access_work'] == 'Si').mean() * 100
    ]
})

# ACTUALIZACIÓN: Usamos .map() en lugar de .applymap() para compatibilidad con Pandas 2.1+
display(comparativa.set_index('Métrica').round(2).style.map(
    lambda x: 'background-color: #d4edda; color: black; font-weight: bold' if x > 0 else '', 
    subset=['Train Estratificado (Nuestro)', 'Test Estratificado (Nuestro)']
))

print("\n" + "="*75)
print("🎯 CONCLUSIÓN: El Split Estratificado calcó perfectamente la realidad del universo.")
print("Guardando set de entrenamiento y prueba oficiales (eliminando la llave auxiliar)...")

# Limpiamos la llave auxiliar que ya cumplió su propósito
train_final = train_strat.drop(columns=['Llave_Estratificacion'])
test_final = test_strat.drop(columns=['Llave_Estratificacion'])

# Guardamos los CSV
train_final.to_csv('C:/Users/Matiu/Documents/GitHub/Data-Science/Futura/Módulo 3/Exam/data_processed/train_ecommerce.csv', index=False)
test_final.to_csv('C:/Users/Matiu/Documents/GitHub/Data-Science/Futura/Módulo 3/Exam/data_processed/test_ecommerce.csv', index=False)

print("✅ Archivos 'train_ecommerce.csv' y 'test_ecommerce.csv' guardados correctamente.")

🔑 1. CREANDO LLAVE COMPUESTA DE ESTRATIFICACIÓN...
Distribución de la Llave Compuesta en el universo total:


Llave_Estratificacion
0_Lab:No    41.22
1_Lab:No    31.92
1_Lab:Si    23.70
0_Lab:Si     3.16
Name: proportion, dtype: float64

---------------------------------------------------------------------------

⚖️ 2. COMPARATIVA DE ROBUSTEZ MUESTRAL (MÉTODO ALEATORIO VS ESTRATIFICADO)


,Universo Real,Train Aleatorio,Test Aleatorio,Train Estratificado (Nuestro),Test Estratificado (Nuestro)
Métrica,,,,,
% Clúster 1 (Motor Comercial),55.620000,55.920000,54.400000,55.630000,55.590000
% Clúster 0 (Consumidor Cauto),44.380000,44.080000,45.600000,44.370000,44.410000
% Acceso Corporativo (Sí),26.860000,26.740000,27.350000,26.860000,26.850000



🎯 CONCLUSIÓN: El Split Estratificado calcó perfectamente la realidad del universo.
Guardando set de entrenamiento y prueba oficiales (eliminando la llave auxiliar)...
✅ Archivos 'train_ecommerce.csv' y 'test_ecommerce.csv' guardados correctamente.
